# Why PCA ?

In [ ]:
!wget -q https://github.com/PSAM-5020-2026S-A/5020-utils/raw/main/src/data_utils.py

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from data_utils import object_from_json_url

### Get Data

In [ ]:
PENGUIN_URL = "https://raw.githubusercontent.com/PSAM-5020-2026S-A/5020-utils/refs/heads/main/datasets/json/penguins.json"
penguin_data = object_from_json_url(PENGUIN_URL)

penguins_df = pd.DataFrame.from_records(penguin_data)
penguins_df

### Penguin Example

Explore the penguin data.

Let's encode the species column into integers.
It's a simple encoding, so we can just do it manually using a function and the `DataFrame.apply()` command.

In [ ]:
species = list(penguins_df["species"].unique())

def species_to_label(s):
  return species.index(s)

penguins_df["label"] = penguins_df["species"].apply(species_to_label)

display(penguins_df)
penguins_df.shape

### Scale

In [ ]:
# TODO: separate input features (drop label and species)
penguins_features_df = penguins_df.drop(columns=["label", "species"])

# TODO: Scale data
sclr = StandardScaler().set_output(transform="pandas").fit(penguins_features_df)

feats_std_df = sclr.transform(penguins_features_df)
feats_std_df

### Covariances

If we're trying to get some insight into our data, we can look at covariance tables and some plots.

Now that we have scaled data we can look at covariance tables.

In [ ]:
# TODO: Look at covariances
feats_std_df.cov()

# TODO: Get 2 or 3 most related features


### Plot the Data

We can look at plots of the most correlated features, and of all of the possible pairs of features.

In [ ]:
print("TOP FEATURES")

# TODO: Add the 3 most correlated features here
#       Code assumes data is in: penguins_features_df
top_features = ["flipper_length_mm", "body_mass_g", "bill_length_mm"]

for i,cx in enumerate(top_features):
  for j,cy in enumerate(top_features):
    if j > i:
      plt.scatter(penguins_features_df[cx], penguins_features_df[cy], c=penguins_df["label"])
      plt.xlabel(cx)
      plt.ylabel(cy)
      plt.show()

print("ALL FEATURES")
# TODO: plot all features
#       like above, but going over all features in penguins_features_df

for i,cx in enumerate(penguins_features_df.columns):
  for j,cy in enumerate(penguins_features_df.columns):
    if j > i:
      plt.scatter(penguins_features_df[cx], penguins_features_df[cy], c=penguins_df["label"])
      plt.xlabel(cx)
      plt.ylabel(cy)
      plt.show()


### PCA

The plots tell us a lot, but information is spread through many images.

The top-correlated features actually make it hard to see the separation between $2$ of the species of penguins.

We can try to simplify how we visualize this data by performing `PCA` and combining some of the original features into _principal components_.

In [ ]:
# TODO: create PCA with 3 components
pca = PCA(n_components=3).set_output(transform="pandas").fit(feats_std_df)

# TODO: fit+transform
penguins_pca_df = pca.transform(feats_std_df)

# TODO: look at explained variance
print(sum(pca.explained_variance_ratio_))

### Covariances Again

Can look at covariance table of the `PCA` data.

In [ ]:
# TODO: Look at covariance of PCA data
penguins_pca_df.cov()
pca.inverse_transform(penguins_pca_df).cov()

Hmmm... the covariances of the `PCA` data look strange...

But, that's actually what's expected.

`PCA` separates our data into new features that are combinations of the previous features, but that are themselves not related to each other.

### Plots

In [ ]:
pca_column_names = penguins_pca_df.columns

# First 2 PCs
plt.scatter(penguins_pca_df[pca_column_names[0]], penguins_pca_df[pca_column_names[1]], c=penguins_df["label"])
plt.xlabel(pca_column_names[0])
plt.ylabel(pca_column_names[1])
plt.title("2 PCs")
plt.show()

# First 3 PCs
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(projection='3d')

ax.scatter(penguins_pca_df[pca_column_names[0]],
           penguins_pca_df[pca_column_names[1]],
           penguins_pca_df[pca_column_names[2]],
           c=penguins_df["label"])
ax.set_xlabel(pca_column_names[0])
ax.set_ylabel(pca_column_names[1])
ax.set_zlabel(pca_column_names[2])
ax.set_title("3 PCs")
plt.show()

Although it has combined some of the features, we can still see a lot of information from our original data.

### Clustering

I wonder what clustering would do...

In [ ]:
penguin_clusterer = KMeans(n_clusters=6).set_output(transform="pandas")
penguin_clusters = penguin_clusterer.fit_predict(penguins_pca_df)

In [ ]:
# First 3 PCs
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(projection='3d')

ax.scatter(penguins_pca_df[pca_column_names[0]],
           penguins_pca_df[pca_column_names[1]],
           penguins_pca_df[pca_column_names[2]],
           c=penguin_clusters)
ax.set_xlabel(pca_column_names[0])
ax.set_ylabel(pca_column_names[1])
ax.set_zlabel(pca_column_names[2])
ax.set_title("3 PCs")
plt.show()